<a href="https://colab.research.google.com/github/kimve1969/mgu_ds_cours/blob/main/semestr%201.4/accounts_transactions.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

# ВМК МГУ, Курс по Data Science (DS) и Machine Learning (ML) 2026 год
1 семестр, Программирование на языке Python,
Проектное задание, банковские счета,
Ким Виталий Евгеньевич, ПАО "Сургутнефтегаз"
14.07.2026

Постановка задачи

1. Запустить приведенную программу
2. Добавить в каждый файл 5-8 записей и проверить работу
3. Сохранить счета и пользователей в виде списков
4. Для пользователей сохраненных в виде списков провести 10-15 транзакций различного вида  вывести результат на экран
5. Добавить метод сохранения списка счетов в файл
6. Добавить лимиты по счетам: минимальный и максимальный остаток и при отслеживание лимитов при проведении транзакций

In [7]:
# Filename: classes.py
import pandas as pd

class Account:
  ''' суперкласс для банковского счета '''
  # лимиты
  nMIN_LIMIT = -1000.00;  # ниже тратить нельзя!
  nMAX_LIMIT = +15000.00; # выше банк не принимает!

  def __init__(self, nAccountNo, nCustomerName, nBalance):
      ''' Конструктор '''
      self.__AccountNo = nAccountNo
      self.__CustomerName = nCustomerName
      self.__Balance = nBalance

  def getAccountNo(self):
      ''' получить номер счета '''
      return self.__AccountNo

  def getCustomerName(self):
      ''' получить имя клиента '''
      return self.__CustomerName

  def getBalance(self):
      ''' получить баланс '''
      return self.__Balance

  def setBalance(self, newBalance):
      ''' установить баланс '''
      self.__Balance = newBalance

  def deposit(self, amount):
      ''' внести депозит '''
      # нельзя внести на депозит больше максимального лимита!
      if self.__Balance + amount > self.nMAX_LIMIT:
        self.__Balance = self.nMAX_LIMIT
        print('БАНК БОЛЬШЕ НЕ ПРИНИМАЕТ! ВАШ БАЛАНС:', self.getBalance(), ' + % ЗА МЕСЯЦ.')
      else:
        self.__Balance = self.__Balance + amount

  def withdraw(self, amount):
      ''' снять деньги '''
      # никак нельзя снять ниже установленного порога!
      if self.__Balance - amount < self.nMIN_LIMIT:
        self.__Balance = self.nMIN_LIMIT
        print('ВЫ ПРЕВЫСИЛИ ЛИМИТ! ВАШ БАЛАНС:', self.getBalance() )
      else:
        self.__Balance = self.__Balance - amount

  def display(self):
      ''' информация по счету '''
      print("Номер счета:", self.__AccountNo)
      print("Клиент:", self.__CustomerName)
      print("Баланс: ${0:.2f}".format(self.__Balance))

class SavingAccount(Account):  # inheritance
  ''' сохранить '''

  def __init__(self, nAccountNo, nCustomerName, nBalance):
      ''' конструктор '''
      super().__init__(nAccountNo, nCustomerName, nBalance)
      self.__interest = 0.01 / 12

  def display_monthly_statement(self):
      self.setBalance(self.getBalance() * (1 + self.__interest))
      print("Ежемесячная выписка по сберегательному счету ")
      super().display()


class CurrentAccount(Account):
  ''' Аккаунт клиента '''

  def __init__(self, nAccountNo, nCustomerName, nBalance):
      ''' конструктор '''
      super().__init__(nAccountNo, nCustomerName, nBalance)

  def display_monthly_statement(self):
      print("Ежемесячный отчет по текущему счету ")
      super().display()

# сохранение счетов в файл

def to_csv(list_accounts, file_name = 'new_accounts.csv'):
  df = pd.DataFrame({'AccountNo':[],'CustomerName':[],'Balance':[]})
  for a in list_accounts:
    new_df = pd.DataFrame({'AccountNo':[a.getAccountNo()], 'CustomerName':a.getCustomerName(), 'Balance':a.getBalance()})
    df = pd.concat([df, new_df], ignore_index=True)
  print('сохраняем счета в файл: ', file_name, '\n')
  df.to_csv(file_name)
  print(df.head(10))


# Загружаем данные по счетам и транзакциям из GitHub и сохраняем в файлы

In [8]:
import urllib.request

urllib.request.urlretrieve('https://raw.githubusercontent.com/kimve1969/mgu_ds_cours/refs/heads/main/semestr%201.4/accounts.dat', 'accounts.dat')
urllib.request.urlretrieve('https://raw.githubusercontent.com/kimve1969/mgu_ds_cours/refs/heads/main/semestr%201.4/transactions.dat', 'transactions.dat')

('transactions.dat', <http.client.HTTPMessage at 0x7de74aa168a0>)

In [9]:
#from classes import *

# открыть файлы для ввода
account_file = open("accounts.dat", 'r')
transaction_file = open("transactions.dat", 'r')

# прочитать все строки из файла аккаунта
account_lines = account_file.readlines()

# прочитать все строки из файла транзакции
transaction_lines = transaction_file.readlines()

# список счетов
list_accounts = []

# просмотреть все учетные записи
for account_line in account_lines:
    # получить данные аккаунта
    account_no = account_line[:6]
    customer_name = account_line[6:35]
    balance = float(account_line[35:])
    if account_no[0] == 'S':
        # создать объект сохранения аккаунта
        account = SavingAccount(account_no, customer_name, balance)
    else:
        # создать объект текущего счета
        account = CurrentAccount(account_no, customer_name, balance)
    # account.display()
    # сохраняем счета целиком
    list_accounts.append(account)

    # пройти через все транзакции
    for transaction_line in transaction_lines:
        # получить детали транзакции
        transaction_date = transaction_line[:8]
        transaction_account = transaction_line[8:14]
        transaction_type = transaction_line[14:15]
        transaction_amount = float(transaction_line[15:])
        # если соответствующий аккаунт
        if transaction_account == account_no:
            if transaction_type == 'D':  # депозит
                account.deposit(transaction_amount)
            else:  # списание
                account.withdraw(transaction_amount)

    # вывод ежемесячного отчета
    account.display_monthly_statement()
    print()

# close files
account_file.close()
transaction_file.close()


Ежемесячный отчет по текущему счету 
Номер счета: C00005
Клиент: Robert Yeo                   
Баланс: $-144.62

Ежемесячный отчет по текущему счету 
Номер счета: C00008
Клиент: Lim Ah Seng                  
Баланс: $3326.37

Ежемесячная выписка по сберегательному счету 
Номер счета: S00001
Клиент: Lim Ah Seng                  
Баланс: $680.72

Ежемесячная выписка по сберегательному счету 
Номер счета: S00002
Клиент: Tan Ah Lian                  
Баланс: $5814.52

Ежемесячный отчет по текущему счету 
Номер счета: C00100
Клиент: Ivanov Petr                  
Баланс: $-500.00

Ежемесячная выписка по сберегательному счету 
Номер счета: S00100
Клиент: Ivanov Petr                  
Баланс: $2001.67



# Пополнение и расход по счетам с учетом выставленных лимитов
1. Лимит на расход по Current Account, можно снять в долг не более 1000 ед.
2. Лимит на пополнение по Saving Account, баланс может быть не более 15 000 ед. + проценты

In [10]:
print('------------------- ЕЩЕ ТРАНЗАКЦИИ И ПРОВЕРЯЕМ НА ЛИМИТЫ ---------------------------\n')
# дополнительные транзакции по счетам
for account in list_accounts:
  if account.getAccountNo()[0]=='S':
    account.deposit(10000)
  else:
    account.withdraw(1000)
  account.display_monthly_statement()
  print()
  #account.deposit()

------------------- ЕЩЕ ТРАНЗАКЦИИ И ПРОВЕРЯЕМ НА ЛИМИТЫ ---------------------------

ВЫ ПРЕВЫСИЛИ ЛИМИТ! ВАШ БАЛАНС: -1000.0
Ежемесячный отчет по текущему счету 
Номер счета: C00005
Клиент: Robert Yeo                   
Баланс: $-1000.00

Ежемесячный отчет по текущему счету 
Номер счета: C00008
Клиент: Lim Ah Seng                  
Баланс: $2326.37

Ежемесячная выписка по сберегательному счету 
Номер счета: S00001
Клиент: Lim Ah Seng                  
Баланс: $10689.62

БАНК БОЛЬШЕ НЕ ПРИНИМАЕТ! ВАШ БАЛАНС: 15000.0  + % ЗА МЕСЯЦ.
Ежемесячная выписка по сберегательному счету 
Номер счета: S00002
Клиент: Tan Ah Lian                  
Баланс: $15012.50

ВЫ ПРЕВЫСИЛИ ЛИМИТ! ВАШ БАЛАНС: -1000.0
Ежемесячный отчет по текущему счету 
Номер счета: C00100
Клиент: Ivanov Petr                  
Баланс: $-1000.00

Ежемесячная выписка по сберегательному счету 
Номер счета: S00100
Клиент: Ivanov Petr                  
Баланс: $12011.67



# TECT ПРОГРАММЫ

In [11]:
# ТЕСТЫ классов
test_account_current = CurrentAccount(1,'TestCurrentAccount',0)
test_accout_saving   = SavingAccount(2,'TestSavingAccount',0)

# проверка на корректность выполнения операция не превосходящих лимит
print('\n-------------------- TEST:------------------------------\n')
test_account_current.deposit(100)
test_account_current.withdraw(50)
test_account_current.display_monthly_statement()
fcheck = test_account_current.getBalance()
assert fcheck == 50, f'Error balance current account! {fcheck}'

test_accout_saving.deposit(100)
test_accout_saving.withdraw(50)
test_accout_saving.display_monthly_statement()
fcheck = abs(test_accout_saving.getBalance()-50)
# проверка баланса с учетом начисленных процентов
# и с учетом точности до 5 знака ...
assert fcheck < (50.0*0.01/12.0 + 0.00001), f'Error balance saving account! {fcheck}'
print()

# проверка на корректность выполнения операций с учетом лимитов
test_account_current.withdraw(1000)
fcheck = test_account_current.getBalance()
assert fcheck != -1000.00, f'Error Min limit balance! {fcheck}'

test_accout_saving.deposit(15000)
fcheck = test_accout_saving.getBalance()
assert  fcheck == 15000.00, f'Error Max limit balance! {fcheck}'

print('\n----------------- TEST OK! ---------------------\n')


-------------------- TEST:------------------------------

Ежемесячный отчет по текущему счету 
Номер счета: 1
Клиент: TestCurrentAccount
Баланс: $50.00
Ежемесячная выписка по сберегательному счету 
Номер счета: 2
Клиент: TestSavingAccount
Баланс: $50.04

БАНК БОЛЬШЕ НЕ ПРИНИМАЕТ! ВАШ БАЛАНС: 15000.0  + % ЗА МЕСЯЦ.

----------------- TEST OK! ---------------------



# Сохраняем счета в CSV-файл

In [12]:
# сохраняем счета в csv-файл
to_csv(list_accounts)

сохраняем счета в файл:  new_accounts.csv 

  AccountNo                   CustomerName       Balance
0    C00005  Robert Yeo                     -1000.000000
1    C00008  Lim Ah Seng                     2326.370000
2    S00001  Lim Ah Seng                    10689.617389
3    S00002  Tan Ah Lian                    15012.500000
4    C00100  Ivanov Petr                    -1000.000000
5    S00100  Ivanov Petr                    12011.668056
